In [2]:
%pip install geopandas pandas requests matplotlib

  Using cached geopandas-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached pandas-3.0.6-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached matplotlib-3.11.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (80 kB)
  Using cached numpy-2.5.3-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-macosx_12_0_arm64.whl.metadata (5.8 kB)
  Using cached pyproj-3.8.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (40 kB)
  Using cached shapely-2.1.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached charset_normalizer-3.5.1-cp314-cp314-macosx_10_15_universal2.whl.metadata (45 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached contourpy-1.4.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (3.8 kB)
  Using cache

In [1]:
import os
import glob
import zipfile
import geopandas as gpd
import pandas as pd

# 1. Buscar automáticamente archivos .zip o .shp en TODAS las subcarpetas
archivos_zip = glob.glob('**/*.zip', recursive=True)
archivos_shp = glob.glob('**/*.shp', recursive=True)

# Si no hay .shp visible pero hay un .zip, lo descomprimimos automáticamente
if not archivos_shp and archivos_zip:
    ruta_zip = archivos_zip[0]
    print(f"📦 Encontrado archivo comprimido en: {ruta_zip}")
    carpeta_destino = os.path.dirname(ruta_zip) if os.path.dirname(ruta_zip) else '.'
    with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
        zip_ref.extractall(carpeta_destino)
    print("✅ ¡Descomprimido con éxito!")
    # Volvemos a buscar el .shp recién extraído
    archivos_shp = glob.glob('**/*.shp', recursive=True)

# 2. Cargar y procesar el Shapefile encontrado
if archivos_shp:
    ruta_shp = archivos_shp[0]
    print(f"✅ ¡Cargando mapa desde: {ruta_shp}!")
    
    parcelas = gpd.read_file(ruta_shp)
    
    # Convertir a coordenadas Latitud / Longitud (WGS84)
    parcelas_latlon = parcelas.to_crs(epsg=4326)
    parcelas_latlon['latitud'] = parcelas_latlon.geometry.centroid.y
    parcelas_latlon['longitud'] = parcelas_latlon.geometry.centroid.x
    
    print("\n🎉 ¡LOGRADO! Aquí están tus primeras parcelas con sus coordenadas:")
    print(parcelas_latlon.head())
else:
    print("❌ No se encontró ningún archivo .zip o .shp en el proyecto.")

✅ ¡Cargando mapa desde: Parcelas_Reto_AGC_CONJUNTO.shp!

🎉 ¡LOGRADO! Aquí están tus primeras parcelas con sus coordenadas:
  Cultivo ID_POLIGON  área_ha              Municipio   Estado       CONJUNTO  \
0  Cebada    AGC_196     8.57        Emiliano Zapata  Hidalgo  ENTRENAMIENTO   
1  Cebada    AGC_044     0.68        Emiliano Zapata  Hidalgo  ENTRENAMIENTO   
2  Cebada    AGC_020     8.34        Emiliano Zapata  Hidalgo  ENTRENAMIENTO   
3  CEBADA    AGC_174     6.89  Cuautepec de Hinojosa  Hidalgo     PREDICCION   
4  CEBADA    AGC_083     2.87  Cuautepec de Hinojosa  Hidalgo  ENTRENAMIENTO   

                                            geometry    latitud   longitud  
0  POLYGON Z ((-98.57739 19.65802 0, -98.57751 19...  19.656257 -98.579074  
1  POLYGON Z ((-98.58655 19.62943 0, -98.5862 19....  19.629537 -98.586252  
2  POLYGON Z ((-98.58494 19.6278 0, -98.58656 19....  19.627922 -98.586114  
3  POLYGON Z ((-98.30604 19.9 0, -98.30486 19.899...  19.898184 -98.304278  
4  POLYGON 

/var/folders/vv/kjhcncmd5wv8qsg2t2kqdrkr0000gn/T/ipykernel_22422/1693030207.py:31: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parcelas_latlon['latitud'] = parcelas_latlon.geometry.centroid.y
/var/folders/vv/kjhcncmd5wv8qsg2t2kqdrkr0000gn/T/ipykernel_22422/1693030207.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parcelas_latlon['longitud'] = parcelas_latlon.geometry.centroid.x


In [2]:
import os
import glob
import zipfile
import geopandas as gpd
import pandas as pd
import requests
import time

# 1. CARGAR Y PROCESAR EL SHAPEFILE AUTOMÁTICAMENTE
print("📂 Buscando y cargando mapa de parcelas...")
archivos_zip = glob.glob('**/*.zip', recursive=True)
archivos_shp = glob.glob('**/*.shp', recursive=True)

if not archivos_shp and archivos_zip:
    ruta_zip = archivos_zip[0]
    carpeta_destino = os.path.dirname(ruta_zip) if os.path.dirname(ruta_zip) else '.'
    with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
        zip_ref.extractall(carpeta_destino)
    archivos_shp = glob.glob('**/*.shp', recursive=True)

ruta_shp = archivos_shp[0]
parcelas = gpd.read_file(ruta_shp)

# Convertir coordenadas a Latitud / Longitud (WGS84)
parcelas_latlon = parcelas.to_crs(epsg=4326)
parcelas_latlon['latitud'] = parcelas_latlon.geometry.centroid.y
parcelas_latlon['longitud'] = parcelas_latlon.geometry.centroid.x

# 2. DESCARGAR CLIMA DE TODAS LAS PARCELAS
resumen_clima_completo = []
total_parcelas = len(parcelas_latlon)

print(f"🚀 Iniciando descarga de clima para las {total_parcelas} parcelas...")

for index, fila in parcelas_latlon.iterrows():
    id_p = fila['ID_POLIGON']
    lat = fila['latitud']
    lon = fila['longitud']
    
    # Imprimir avance cada 10 parcelas
    if (index + 1) % 10 == 0 or (index + 1) == total_parcelas:
        print(f"⏳ Procesando parcela {index + 1} de {total_parcelas} ({id_p})...")
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    parametros = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2024-05-01",
        "end_date": "2024-10-31",
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "et0_fao_evapotranspiration"
        ],
        "timezone": "America/Mexico_City"
    }
    
    try:
        respuesta = requests.get(url, params=parametros).json()
        df_diario = pd.DataFrame(respuesta['daily'])
        
        # Calcular Grados Días de Desarrollo (GDD) base 5°C
        df_diario['GDD'] = ((df_diario['temperature_2m_max'] + df_diario['temperature_2m_min']) / 2) - 5.0
        df_diario['GDD'] = df_diario['GDD'].apply(lambda x: max(0, x))
        
        # Guardar resumen biofísico
        resumen_clima_completo.append({
            'ID_POLIGON': id_p,
            'GDD_acumulados_2024': df_diario['GDD'].sum(),
            'Lluvia_total_mm_2024': df_diario['precipitation_sum'].sum(),
            'ETo_acumulada_mm_2024': df_diario['et0_fao_evapotranspiration'].sum(),
            'Deficit_hidrico_mm_2024': df_diario['precipitation_sum'].sum() - df_diario['et0_fao_evapotranspiration'].sum()
        })
    except Exception as e:
        print(f"⚠️ Error en la parcela {id_p}: {e}")
    
    time.sleep(0.05)

# 3. GUARDAR TABLA EN ARCHIVO CSV
df_resumen_final = pd.DataFrame(resumen_clima_completo)
nombre_csv = 'resumen_clima_parcelas_2024.csv'
df_resumen_final.to_csv(nombre_csv, index=False)

print(f"\n🎉 ¡PROCESO FINALIZADO CON ÉXITO!")
print(f"📁 Se guardó el archivo '{nombre_csv}' con {len(df_resumen_final)} filas.")

📂 Buscando y cargando mapa de parcelas...


/var/folders/vv/kjhcncmd5wv8qsg2t2kqdrkr0000gn/T/ipykernel_22797/1163693464.py:26: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parcelas_latlon['latitud'] = parcelas_latlon.geometry.centroid.y
/var/folders/vv/kjhcncmd5wv8qsg2t2kqdrkr0000gn/T/ipykernel_22797/1163693464.py:27: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parcelas_latlon['longitud'] = parcelas_latlon.geometry.centroid.x


🚀 Iniciando descarga de clima para las 197 parcelas...
⏳ Procesando parcela 10 de 197 (AGC_064)...
⏳ Procesando parcela 20 de 197 (AGC_065)...
⏳ Procesando parcela 30 de 197 (AGC_003)...
⏳ Procesando parcela 40 de 197 (AGC_030)...
⏳ Procesando parcela 50 de 197 (AGC_118)...
⏳ Procesando parcela 60 de 197 (AGC_097)...
⏳ Procesando parcela 70 de 197 (AGC_177)...
⏳ Procesando parcela 80 de 197 (AGC_018)...
⏳ Procesando parcela 90 de 197 (AGC_116)...
⏳ Procesando parcela 100 de 197 (AGC_015)...
⏳ Procesando parcela 110 de 197 (AGC_127)...
⏳ Procesando parcela 120 de 197 (AGC_168)...
⏳ Procesando parcela 130 de 197 (AGC_094)...
⏳ Procesando parcela 140 de 197 (AGC_049)...
⏳ Procesando parcela 150 de 197 (AGC_193)...
⏳ Procesando parcela 160 de 197 (AGC_112)...
⏳ Procesando parcela 170 de 197 (AGC_053)...
⏳ Procesando parcela 180 de 197 (AGC_173)...
⏳ Procesando parcela 190 de 197 (AGC_037)...
⏳ Procesando parcela 197 de 197 (AGC_001)...

🎉 ¡PROCESO FINALIZADO CON ÉXITO!
📁 Se guardó el archi